This is the notebook to define the 8 Qubit QAOA in Bloqade SDK using python:

We need Hc, which is the 8 values that come out of H function after passing it the feateures variables

We need Hm, which we define it, a common way to define it is using RX gate (rotations in X)

We need to define the QAOA algorithm, that in essence is: 
- Starting the 8 qubits
- Apply Hadamard gates to create equal superposition
- Apply Hc part of the circuit which would be quantum gates 
- Apply Hm part of the circuit which would be quantum gates
- Measure it   

** After measurement you get a bitstring **

After having your QAOA defined, we also have to define auxiliar functions in python to make x quantity of shots per circuit, define the probability of a bitstring, the function to calculate energy/cost of a bitstring, a classical optimizer to update parameters from QAOA, run the circuit again, so we need the function to run the circuit

**pip installs**

In [1]:
pip install -q bloqade 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
!{sys.executable} -m pip install -U lark

**Imports**

In [3]:
import math
from typing import Any
import numpy as np 
import kirin
from kirin.dialects import ilist

from bloqade import qasm2

pi = math.pi

**Global variables from Hamiltonian function we produced**

With these variables produced already, we do not need to use the dataset again or touch it in this ipynb because we already produced the hamiltonian cost on the variables we are using.

The ground truth was produced by brute forcing the 256 combinations and checking the best one, since is easy for a classical computer to do that 

In [4]:
n_qubits = 8

qubit_labels = [
    "A004_LSB", "A004_MSB",
    "A047_LSB", "A047_MSB",
    "A026_LSB", "A026_MSB",
    "A017_LSB", "A017_MSB",
]

# Constant offset is irrelevant for QAOA optimization because it does not change
# which bitstring minimizes the objective.
c0 = 0.0

# Local fields h_i for H_C = sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
h_terms = [
    (0, -0.048631),
    (1, -0.093696),
    (2, -0.036724),
    (3, -0.071445),
    (4, -0.070662),
    (5, -0.133323),
    (6, -0.112826),
    (7, -0.203428),
]

# Pair couplings J_ij
J_terms = [
    (0, 1, 0.003566),
    (0, 2, 0.001333),
    (0, 3, 0.002667),
    (0, 4, 0.002667),
    (0, 5, 0.005334),
    (0, 6, 0.004444),
    (0, 7, 0.008889),
    (1, 2, 0.002667),
    (1, 3, 0.005334),
    (1, 4, 0.005334),
    (1, 5, 0.010667),
    (1, 6, 0.008889),
    (1, 7, 0.017778),
    (2, 3, 0.002003),
    (2, 4, 0.002000),
    (2, 5, 0.004000),
    (2, 6, 0.003333),
    (2, 7, 0.006667),
    (3, 4, 0.004000),
    (3, 5, 0.008000),
    (3, 6, 0.006667),
    (3, 7, 0.013334),
    (4, 5, 0.008002),
    (4, 6, 0.006667),
    (4, 7, 0.013334),
    (5, 6, 0.013334),
    (5, 7, 0.026667),
    (6, 7, 0.022225),
]

ground_truth_bitstring = "11011110"

**First we are going to convert the Hamiltonian data to arrays**

In [5]:
import numpy as np

# Build dense h vector and J matrix from the term lists
h8 = np.zeros(n_qubits, dtype=float)
J8 = np.zeros((n_qubits, n_qubits), dtype=float)

for i, hi in h_terms:
    h8[i] = hi

for i, j, Jij in J_terms:
    J8[i, j] = Jij
    J8[j, i] = Jij

ising_offset = 0.0

print("h8:")
print(h8)
print("\nJ8:")
print(J8)

h8:
[-0.048631 -0.093696 -0.036724 -0.071445 -0.070662 -0.133323 -0.112826
 -0.203428]

J8:
[[0.       0.003566 0.001333 0.002667 0.002667 0.005334 0.004444 0.008889]
 [0.003566 0.       0.002667 0.005334 0.005334 0.010667 0.008889 0.017778]
 [0.001333 0.002667 0.       0.002003 0.002    0.004    0.003333 0.006667]
 [0.002667 0.005334 0.002003 0.       0.004    0.008    0.006667 0.013334]
 [0.002667 0.005334 0.002    0.004    0.       0.008002 0.006667 0.013334]
 [0.005334 0.010667 0.004    0.008    0.008002 0.       0.013334 0.026667]
 [0.004444 0.008889 0.003333 0.006667 0.006667 0.013334 0.       0.022225]
 [0.008889 0.017778 0.006667 0.013334 0.013334 0.026667 0.022225 0.      ]]


**Classical Icing Energy**

In [6]:
def ising_energy_from_x_list(x_list, h, J, ising_offset=0.0):
    # Use the convention x = (1 + s)/2, so s = 2x - 1
    s = np.array([2 * int(bit) - 1 for bit in x_list], dtype=float)

    energy = ising_offset

    for i in range(len(h)):
        energy += h[i] * s[i]

    for i in range(J.shape[0]):
        for j in range(i + 1, J.shape[1]):
            energy += J[i, j] * s[i] * s[j]

    return float(energy)

**Helper to convert bitstring to x-list**

In [7]:
def bitstring_to_x_list(bitstring):
    return [int(b) for b in bitstring]

**Decode 2 bits per asset**

In [8]:
def decode_lots_from_bitstring(bitstring):
    """
    Decode 8 qubits into 4 asset lot values using:
    lot = LSB + 2 * MSB
    """
    bits = [int(b) for b in bitstring]
    lots = []

    for k in range(0, len(bits), 2):
        lsb = bits[k]
        msb = bits[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)

    return lots

**Verify the portfolio**

In [9]:
decoded_lots = decode_lots_from_bitstring(ground_truth_bitstring)
print("Decoded lots from ground-truth bitstring:", decoded_lots)

Decoded lots from ground-truth bitstring: [3, 2, 3, 1]


**Build the QAOA algorithm on Bloqade**


In [10]:
def build_qaoa_ising_kernel(n_qubits, h_terms, J_terms):
    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(n_qubits)

        # Initialize the register in the |+> state
        for i in range(n_qubits):
            qasm2.h(q[i])

        # Apply p QAOA layers
        for layer in range(len(gamma)):
            g = gamma[layer]
            b = beta[layer]

            # Single-qubit Z terms
            for k in range(len(h_terms)):
                term = h_terms[k]
                i = term[0]
                hi = term[1]
                qasm2.rz(q[i], 2.0 * g * hi)

            # Two-qubit ZZ terms
            for k in range(len(J_terms)):
                term = J_terms[k]
                i = term[0]
                j = term[1]
                Jij = term[2]

                qasm2.cx(q[i], q[j])
                qasm2.rz(q[j], 2.0 * g * Jij)
                qasm2.cx(q[i], q[j])

            # Mixer layer
            for i in range(n_qubits):
                qasm2.rx(q[i], 2.0 * b)

        return q

    return kernel

**Create the Kernel**

In [11]:
kernel = build_qaoa_ising_kernel(n_qubits, h_terms, J_terms)
print(kernel)

Method("kernel")


In [12]:
print(kernel)
print(type(kernel))

Method("kernel")
<class 'kirin.ir.method.Method'>


**Choose initial QAOA parameters**

In [13]:
# P stands for the depth of the circuit, how many times we repeat the gates
p = 1

gamma_init = [0.3] * p
beta_init = [0.2] * p

**wrap in main and emit QASM**

In [14]:
@qasm2.extended
def main():
    return kernel(gamma_init, beta_init)

**print the generated circuit**

In [15]:
target = qasm2.emit.QASM2()
ast = target.emit(main)
qasm2.parse.pprint(ast)

OPENQASM 2.0;
include "qelib1.inc";
qreg q[8];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];
h q[7];
rz (-0.0291786) q[0];
rz (-0.0562176) q[1];
rz (-0.0220344) q[2];
rz (-0.042866999999999995) q[3];
rz (-0.0423972) q[4];
rz (-0.07999379999999999) q[5];
rz (-0.0676956) q[6];
rz (-0.1220568) q[7];
CX q[0], q[1];
rz (0.0021396) q[1];
CX q[0], q[1];
CX q[0], q[2];
rz (0.0007997999999999999) q[2];
CX q[0], q[2];
CX q[0], q[3];
rz (0.0016002) q[3];
CX q[0], q[3];
CX q[0], q[4];
rz (0.0016002) q[4];
CX q[0], q[4];
CX q[0], q[5];
rz (0.0032004) q[5];
CX q[0], q[5];
CX q[0], q[6];
rz (0.0026663999999999998) q[6];
CX q[0], q[6];
CX q[0], q[7];
rz (0.005333399999999999) q[7];
CX q[0], q[7];
CX q[1], q[2];
rz (0.0016002) q[2];
CX q[1], q[2];
CX q[1], q[3];
rz (0.0032004) q[3];
CX q[1], q[3];
CX q[1], q[4];
rz (0.0032004) q[4];
CX q[1], q[4];
CX q[1], q[5];
rz (0.006400199999999999) q[5];
CX q[1], q[5];
CX q[1], q[6];
rz (0.005333399999999999) q[6];
CX q[1], q[6];
CX q[1], q[7];
rz (0.01

**Test the ground truth**

In [16]:
ground_truth_x = [1, 1, 0, 1, 1, 1, 1, 0]

print("Ground-truth x:", ground_truth_x)
print("Ground-truth lots:", decode_lots_from_bitstring("".join(str(b) for b in ground_truth_x)))
print("Ground-truth corrected Ising energy:", ising_energy_from_x_list(ground_truth_x, h8, J8, ising_offset))

Ground-truth x: [1, 1, 0, 1, 1, 1, 1, 0]
Ground-truth lots: [3, 2, 3, 1]
Ground-truth corrected Ising energy: -0.30575499999999994


**Brute-force classical check over all 256 bitstrings**

In [17]:
def x_list_to_lots(x_list):
    lots = []
    for k in range(0, len(x_list), 2):
        lsb = x_list[k]
        msb = x_list[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)
    return lots

all_results_rebuilt = []

for num in range(2 ** n_qubits):
    x_list = [(num >> i) & 1 for i in range(n_qubits)]
    e_ising = ising_energy_from_x_list(x_list, h8, J8, ising_offset)

    all_results_rebuilt.append({
        "x": x_list,
        "lots": x_list_to_lots(x_list),
        "ising_energy": e_ising
    })

all_results_rebuilt = sorted(all_results_rebuilt, key=lambda r: r["ising_energy"])

print("Top 15 solutions ranked by corrected Ising energy:\n")
for rank, r in enumerate(all_results_rebuilt[:15], start=1):
    print(f"{rank:2d}. x={r['x']}  lots={r['lots']}  Ising={r['ising_energy']:.8f}")

Top 15 solutions ranked by corrected Ising energy:

 1. x=[1, 1, 1, 1, 1, 1, 1, 1]  lots=[3, 3, 3, 3]  Ising=-0.55093300
 2. x=[1, 1, 0, 1, 1, 1, 1, 1]  lots=[3, 2, 3, 3]  Ising=-0.52149100
 3. x=[0, 1, 1, 1, 1, 1, 1, 1]  lots=[2, 3, 3, 3]  Ising=-0.51147100
 4. x=[1, 1, 1, 1, 0, 1, 1, 1]  lots=[3, 3, 2, 3]  Ising=-0.49361700
 5. x=[1, 1, 1, 0, 1, 1, 1, 1]  lots=[3, 1, 3, 3]  Ising=-0.49205300
 6. x=[0, 1, 0, 1, 1, 1, 1, 1]  lots=[2, 2, 3, 3]  Ising=-0.47669700
 7. x=[1, 0, 1, 1, 1, 1, 1, 1]  lots=[1, 3, 3, 3]  Ising=-0.47201100
 8. x=[1, 1, 1, 1, 1, 1, 0, 1]  lots=[3, 3, 3, 2]  Ising=-0.45639900
 9. x=[1, 1, 0, 1, 0, 1, 1, 1]  lots=[3, 2, 2, 3]  Ising=-0.45617500
10. x=[1, 1, 0, 0, 1, 1, 1, 1]  lots=[3, 0, 3, 3]  Ising=-0.45459900
11. x=[0, 1, 1, 1, 0, 1, 1, 1]  lots=[2, 3, 2, 3]  Ising=-0.44348700
12. x=[0, 1, 1, 0, 1, 1, 1, 1]  lots=[2, 1, 3, 3]  Ising=-0.44192300
13. x=[1, 1, 1, 1, 1, 0, 1, 1]  lots=[3, 3, 1, 3]  Ising=-0.43629500
14. x=[1, 0, 0, 1, 1, 1, 1, 1]  lots=[1, 2, 3, 3]  

**Check the rank of the known ground truth**

In [18]:
gt_rank = None

for idx, r in enumerate(all_results_rebuilt):
    if r["x"] == ground_truth_x:
        gt_rank = idx + 1
        print("Ground-truth entry found:")
        print(r)
        break

print("\nGround-truth rank under corrected Ising evaluation:", gt_rank)

Ground-truth entry found:
{'x': [1, 1, 0, 1, 1, 1, 1, 0], 'lots': [3, 2, 3, 1], 'ising_energy': -0.30575499999999994}

Ground-truth rank under corrected Ising evaluation: 44
